In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install jupysql
%pip install duckdb-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.0/879.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00


In [ ]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb://

In [ ]:
import duckdb
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir las rutas
dataset_path = '/content/drive/MyDrive/DMEyF/'
dataset_file = 'competencia_01.csv'
ruta_completa = dataset_path + dataset_file

# 3. Conectar a DuckDB
con = duckdb.connect(database=':memory:', read_only=False)

# 4. Cargar el dataset original usando la ruta completa
print(f"Cargando datos desde: {ruta_completa}...")
con.execute(f"""
    CREATE TABLE dataset_original AS
    SELECT * FROM read_csv_auto('{ruta_completa}');
""")

# 5. Ejecutar la ingeniería de características optimizada para LightGBM
print("Calculando features de ingeniería...")
query_features = f"""
CREATE OR REPLACE TABLE churn_features_final AS
SELECT
    *,

    -- === A. INTENSIDAD DIGITAL Y CANALES ===
    (COALESCE(chomebanking_transacciones, 0) + COALESCE(cmobile_app_trx, 0)) /
    NULLIF(CAST(ctrx_quarter AS DOUBLE), 0) AS ratio_digital_trx,

    COALESCE(chomebanking_transacciones, 0) /
    NULLIF(CAST(cmobile_app_trx AS DOUBLE) + COALESCE(chomebanking_transacciones, 0), 0) AS share_homebanking,

    -- === B. UTILIZACIÓN DE LÍNEAS DE CRÉDITO ===
    CASE
        WHEN Visa_mlimitecompra > 0 THEN Visa_msaldototal / Visa_mlimitecompra
        ELSE 0
    END AS visa_utilization_ratio,

    CASE
        WHEN Master_mlimitecompra > 0 THEN Master_msaldototal / Master_mlimitecompra
        ELSE 0
    END AS master_utilization_ratio,

    CASE
        WHEN Visa_mpagominimo > 0 THEN Visa_mpagado / Visa_mpagominimo
        ELSE 1
    END AS visa_ratio_pago_minimo,

    -- === C. PROFUNDIZACIÓN Y DENSIDAD DE PRODUCTOS ===
    cproductos / (cliente_antiguedad + 1.0) AS productos_por_antiguedad,

    -- === D. ESTRÉS FINANCIERO Y RIESGO ===
    COALESCE(ccheques_depositados_rechazados, 0) +
    COALESCE(ccheques_emitidos_rechazados, 0) AS total_cheques_rechazados,

    GREATEST(COALESCE(Visa_delinquency, 0), COALESCE(Master_delinquency, 0)) AS max_delinquency_cards,

    -- === E. RATIOS FINANCIEROS Y MÁRGENES ===
    CASE
        WHEN mactivos_margen > 0 THEN mpasivos_margen / mactivos_margen
        ELSE 0
    END AS ratio_pasivos_activos

FROM dataset_original;
"""

con.execute(query_features)

# 6. Exportar el nuevo dataset enriquecido de vuelta a Google Drive
output_file = dataset_path + 'churn_features_lightgbm.csv'
print(f"Exportando resultado a: {output_file}...")
con.execute(f"""
    COPY churn_features_final TO '{output_file}' (HEADER, DELIMITER ',');
""")

print("¡Proceso finalizado con éxito! El archivo ya está disponible en tu Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando datos desde: /content/drive/MyDrive/DMEyF/competencia_01.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Calculando features de ingeniería...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exportando resultado a: /content/drive/MyDrive/DMEyF/churn_features_lightgbm.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

¡Proceso finalizado con éxito! El archivo ya está disponible en tu Google Drive.
